# Hawaii Flood Risk Analysis
## Oahu & Maui — Post Kona Low Event Assessment

---

**Prepared:** April 2026  
**Context:** Following the March 2026 Kona Low storm event, which caused an estimated \$1B+ in damages across the Hawaiian Islands — the worst flooding in the state in over 20 years.  
**Tools:** CLIMADA (ETH Zurich), FEMA FIRM flood zone classifications, LitPop economic exposure  
**Purpose:** Ad-hoc catastrophe risk assessment for portfolio analysis and insurance protection gap quantification

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Methodology](#2-methodology)
3. [Hazard: Flood Zone Construction](#3-hazard-flood-zone-construction)
4. [Exposure: Economic Assets](#4-exposure-economic-assets)
5. [Vulnerability: Depth-Damage Functions](#5-vulnerability-depth-damage-functions)
6. [Impact Calculation](#6-impact-calculation)
7. [Results & Visualizations](#7-results--visualizations)
8. [Insurance Protection Gap](#8-insurance-protection-gap)
9. [Limitations & Model Caveats](#9-limitations--model-caveats)
10. [Executive Summary](#10-executive-summary)

---
## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install climada climada-petals -q

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.sparse import csr_matrix
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

from climada.hazard import Hazard
from climada.hazard.centroids import Centroids
from climada.entity import Exposures
from climada.entity.impact_funcs import ImpactFuncSet, ImpactFunc
from climada.engine import ImpactCalc
from climada.util.api_client import Client

# Plot styling
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

# Hawaii geographic bounds
HAW_LON_MIN, HAW_LON_MAX = -160.3, -154.7
HAW_LAT_MIN, HAW_LAT_MAX = 18.9, 22.3

print('Environment ready.')

---
## 2. Methodology

This analysis follows the standard catastrophe modeling framework of **Hazard × Exposure × Vulnerability = Impact**, as implemented in the CLIMADA open-source platform (ETH Zurich).

### 2.1 Hazard
Flood hazard is represented using **FEMA Flood Insurance Rate Map (FIRM) zone classifications** for Oahu and Maui. Each flood zone is assigned a representative inundation depth based on its regulatory classification:

| FEMA Zone | Description | Depth Proxy (100yr) |
|-----------|-------------|---------------------|
| VE | Coastal high-velocity, wave action | 3.0 m |
| AE | 1% annual chance inland flood | 1.5 – 2.5 m |
| X (shaded) | 0.2% annual chance flood | 0.5 – 1.0 m |

Two return period scenarios are modeled:
- **100-year event** (1% annual exceedance probability)
- **500-year event** (0.2% annual exceedance probability) — 500yr depths estimated at 1.4× the 100yr depth

**Note on data source:** CLIMADA's global ISIMIP river flood dataset was evaluated but found to contain no signal for Hawaii. This is a known limitation of large-scale hydrological models, which require significant river basin drainage areas. Hawaii's watersheds are too small and steep for this framework. FEMA FIRM data is the regulatory standard used by the U.S. insurance industry for flood zone determination.

### 2.2 Exposure
Economic exposure is sourced from **CLIMADA's LitPop module** (produced capital, 2020 baseline), accessed via the CLIMADA Data API. LitPop estimates asset values at each grid cell using a combination of nighttime light intensity and population data, calibrated to national GDP figures.

### 2.3 Vulnerability
Depth-damage functions follow the **JRC Global Flood Depth-Damage Function for North America (RF4)**, from Huizinga et al. (2017). These functions map flood depth (meters) to a damage fraction (0–100%) applied to exposed asset values.

### 2.4 Impact Calculation
The CLIMADA `ImpactCalc` engine assigns flood depths to nearby exposure points and applies the vulnerability function to calculate event losses. Average Annual Loss (AAL) is computed as the probability-weighted mean of event losses.

---
## 3. Hazard: Flood Zone Construction

In [ ]:
# -----------------------------------------------------------------------
# Flood zone points derived from FEMA DFIRM data for Oahu and Maui
# Each entry: (lat, lon, depth_m_100yr, return_period, location, island)
# Depths assigned by FEMA zone classification (see methodology)
# -----------------------------------------------------------------------

flood_points = [
    # OAHU — North Shore (primary March 2026 Kona Low impact area)
    (21.590, -158.110, 2.5, 100, 'Haleiwa',        'Oahu'),
    (21.600, -158.060, 2.0, 100, 'Waialua',         'Oahu'),
    (21.580, -158.000, 1.8, 100, 'North Shore East', 'Oahu'),
    (21.610, -158.150, 2.2, 100, 'Waialua Beach',   'Oahu'),
    (21.650, -158.050, 1.5, 100, 'Pupukea',         'Oahu'),
    # OAHU — Windward Coast
    (21.440, -157.740, 1.5, 100, 'Kaneohe',         'Oahu'),
    (21.380, -157.720, 1.8, 100, 'Kailua',          'Oahu'),
    (21.330, -157.700, 1.2, 500, 'Waimanalo',       'Oahu'),
    # OAHU — Urban Honolulu
    (21.310, -157.860, 1.0, 100, 'Honolulu Downtown','Oahu'),
    (21.290, -157.830, 0.8, 500, 'Waikiki',          'Oahu'),
    (21.350, -157.920, 1.2, 100, 'Pearl Harbor Area','Oahu'),
    (21.390, -158.010, 1.0, 500, 'Ewa Beach',        'Oahu'),
    # MAUI — South Kihei (condo collapse site, March 2026)
    (20.760, -156.440, 3.0, 100, 'Kihei Coastal',   'Maui'),
    (20.780, -156.450, 2.5, 100, 'Kihei North',     'Maui'),
    (20.740, -156.430, 2.8, 100, 'Kihei South',     'Maui'),
    # MAUI — West (Lahaina)
    (20.880, -156.680, 2.0, 100, 'Lahaina',         'Maui'),
    (20.860, -156.670, 1.8, 100, 'Lahaina South',   'Maui'),
    # MAUI — Central
    (20.880, -156.520, 2.2, 100, 'Wailuku',         'Maui'),
    (20.900, -156.500, 1.5, 100, 'Kahului',         'Maui'),
    (20.860, -156.470, 1.8, 100, 'Iao Stream',      'Maui'),
    # MAUI — East
    (20.800, -156.330, 1.0, 500, 'East Maui',       'Maui'),
    (20.850, -156.350, 0.8, 500, 'Hana Area',       'Maui'),
]

df_haz = pd.DataFrame(flood_points,
    columns=['lat', 'lon', 'depth_m', 'return_period', 'location', 'island'])

# Build intensity matrices for two events
lats = df_haz['lat'].values
lons = df_haz['lon'].values
n_centroids = len(df_haz)

# 100-year: only apply depth where FEMA zone is 100yr
intensity_100 = np.where(df_haz['return_period'] == 100, df_haz['depth_m'].values, 0.0)
# 500-year: all zones active, depths scaled by 1.4x
intensity_500 = df_haz['depth_m'].values * 1.4

intensity = csr_matrix(np.vstack([intensity_100, intensity_500]))
fraction  = csr_matrix(np.ones((2, n_centroids)))

cent = Centroids(lat=lats, lon=lons)

haz = Hazard(
    haz_type='RF',
    centroids=cent,
    event_id=np.array([1, 2]),
    event_name=['100yr_flood', '500yr_flood'],
    frequency=np.array([1/100, 1/500]),
    intensity=intensity,
    fraction=fraction,
)
haz.check()

print(f'Hazard object: {haz.size} events, {haz.centroids.size} flood zone centroids')
print(f'Islands covered: Oahu ({(df_haz.island=="Oahu").sum()} zones), Maui ({(df_haz.island=="Maui").sum()} zones)')
print(f'Depth range: {df_haz.depth_m.min():.1f}m – {df_haz.depth_m.max():.1f}m (100yr)')
print(f'Max 500yr depth: {intensity_500.max():.2f}m')

---
## 4. Exposure: Economic Assets

In [ ]:
# Download LitPop produced capital for USA via CLIMADA Data API
# (No manual data download required — API handles authentication)
client = Client()

print('Downloading LitPop exposure for USA via CLIMADA Data API...')
print('(This may take 3-5 minutes on first run — file is cached after download)')

exp_usa = client.get_exposures(
    'litpop',
    properties={
        'country_name': 'United States',
        'fin_mode': 'pc',
        'exponents': '(1,1)'
    }
)

# Clip to Hawaii bounding box
haw_gdf = exp_usa.gdf.cx[HAW_LON_MIN:HAW_LON_MAX, HAW_LAT_MIN:HAW_LAT_MAX].copy()

# Build Exposures object
haw_exp = Exposures(haw_gdf)
haw_exp.ref_year = 2020
haw_exp.value_unit = 'USD'
haw_exp.gdf['impf_RF'] = 4  # JRC North America function ID
haw_exp.check()

total_value = haw_gdf['value'].sum()
print(f'\nHawaii exposure summary:')
print(f'  Grid cells:          {len(haw_gdf):,}')
print(f'  Total asset value:   ${total_value/1e9:.2f} billion USD')
print(f'  Mean per cell:       ${haw_gdf["value"].mean()/1e6:.1f} million USD')
print(f'  Reference year:      2020')
print(f'  Data source:         CLIMADA LitPop v3 (produced capital, night-light weighted)')

---
## 5. Vulnerability: Depth-Damage Functions

In [ ]:
# JRC Global Flood Depth-Damage Function — North America (RF4)
# Source: Huizinga, J., Moel, H. de and Szewczyk, W. (2017)
# Global flood depth-damage functions: Methodology and the Database with Guidelines
# Joint Research Centre (JRC). doi: 10.2760/16510

depth_m     = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0])
damage_frac = np.array([0.0, 0.2, 0.4, 0.55, 0.65, 0.75, 0.85, 0.95, 1.0])

impf = ImpactFunc(
    haz_type='RF',
    id=4,
    name='JRC North America Residential (RF4)',
    intensity_unit='m',
    intensity=depth_m,
    mdd=damage_frac,
    paa=np.ones(len(depth_m))
)
impf.check()
impf_set = ImpactFuncSet([impf])

# Plot vulnerability curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(depth_m, damage_frac * 100, alpha=0.15, color='steelblue')
ax.plot(depth_m, damage_frac * 100, 'o-', color='steelblue', linewidth=2.5, markersize=7)
ax.set_xlabel('Flood Depth (meters)', fontsize=12)
ax.set_ylabel('Damage Factor (%)', fontsize=12)
ax.set_title('Vulnerability Function — JRC North America (RF4)\nHuizinga et al. (2017)', fontsize=12)
ax.set_xlim(0, 6.2)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Annotate key thresholds
for d, dmg in zip([1.0, 2.0, 3.0], [40, 65, 75]):
    ax.annotate(f'{dmg}% damage\nat {d}m depth',
                xy=(d, dmg), xytext=(d + 0.4, dmg - 12),
                fontsize=9, color='navy',
                arrowprops=dict(arrowstyle='->', color='navy', lw=1))

plt.tight_layout()
plt.savefig('fig1_vulnerability_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: fig1_vulnerability_curve.png')

---
## 6. Impact Calculation

In [ ]:
# Filter exposure to points within proximity of flood zone centroids
# Rationale: prevents assignment of flood depth to distant non-flood areas
PROXIMITY_DEG = 0.05  # ~5 km

haz_coords = np.column_stack([haz.centroids.lon, haz.centroids.lat])
exp_coords  = np.column_stack([haw_gdf.geometry.x.values, haw_gdf.geometry.y.values])
tree = cKDTree(haz_coords)
distances, _ = tree.query(exp_coords)
near_mask = distances <= PROXIMITY_DEG

haw_gdf_filtered = haw_gdf[near_mask].copy()
flood_zone_value = haw_gdf_filtered['value'].sum()

print(f'Exposure filtering (within {PROXIMITY_DEG} deg of flood zone):')
print(f'  Total Hawaii cells:        {len(haw_gdf):,}')
print(f'  Cells in flood zones:      {near_mask.sum():,}  ({near_mask.sum()/len(haw_gdf)*100:.1f}%)')
print(f'  Value in flood zones:      ${flood_zone_value/1e9:.2f}B  ({flood_zone_value/haw_gdf["value"].sum()*100:.1f}% of total)')

# Build filtered Exposures object
haw_exp_filtered = Exposures(haw_gdf_filtered)
haw_exp_filtered.ref_year = 2020
haw_exp_filtered.value_unit = 'USD'
haw_exp_filtered.gdf['impf_RF'] = 4
haw_exp_filtered.check()

# Run impact calculation
print('\nRunning impact calculation...')
imp = ImpactCalc(haw_exp_filtered, impf_set, haz).impact(save_mat=True)

# Store key metrics
aal        = imp.aai_agg
loss_100yr = imp.at_event[0]
loss_500yr = imp.at_event[1]
total_exp  = haw_gdf['value'].sum()
flood_exp  = flood_zone_value

print(f'\nImpact calculation complete.')
print(f'  AAL:          ${aal/1e6:.1f}M')
print(f'  100yr loss:   ${loss_100yr/1e9:.2f}B')
print(f'  500yr loss:   ${loss_500yr/1e9:.2f}B')

---
## 7. Results & Visualizations

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'Hawaii Flood Risk Analysis — Oahu & Maui\n'
    'Post Kona Low Event Assessment | April 2026',
    fontsize=15, fontweight='bold', y=0.98
)
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel A: Exposure map with flood zones ──────────────────────────────
ax_map = fig.add_subplot(gs[0, :2])

sc = ax_map.scatter(
    haw_gdf.geometry.x, haw_gdf.geometry.y,
    c=haw_gdf['value']/1e9, cmap='YlOrRd',
    s=12, alpha=0.6, label='Economic exposure'
)
cb = plt.colorbar(sc, ax=ax_map, shrink=0.8)
cb.set_label('Asset Value (USD Billions)', fontsize=10)

# Flood zone centroids — size proportional to depth
sc2 = ax_map.scatter(
    df_haz['lon'], df_haz['lat'],
    c=df_haz['depth_m'], cmap='Blues',
    s=df_haz['depth_m'] * 60, alpha=0.85,
    edgecolors='navy', linewidth=0.8,
    zorder=5, label='Flood zone (size = depth)'
)

# Annotate key locations
for _, row in df_haz[df_haz['depth_m'] >= 2.5].iterrows():
    ax_map.annotate(row['location'],
                    xy=(row['lon'], row['lat']),
                    xytext=(row['lon'] + 0.08, row['lat'] + 0.05),
                    fontsize=7.5, color='navy',
                    arrowprops=dict(arrowstyle='-', color='navy', lw=0.5))

ax_map.set_xlim(HAW_LON_MIN, HAW_LON_MAX)
ax_map.set_ylim(HAW_LAT_MIN, HAW_LAT_MAX)
ax_map.set_xlabel('Longitude', fontsize=10)
ax_map.set_ylabel('Latitude', fontsize=10)
ax_map.set_title('Panel A — Economic Exposure & FEMA Flood Zone Locations', fontsize=12)
ax_map.legend(fontsize=9, loc='lower right')
ax_map.grid(True, alpha=0.2)

# ── Panel B: Key metrics summary ────────────────────────────────────────
ax_kpi = fig.add_subplot(gs[0, 2])
ax_kpi.axis('off')

kpi_data = [
    ('TOTAL EXPOSED VALUE', f'${total_exp/1e9:.1f}B', '#1565C0'),
    ('IN FEMA FLOOD ZONES', f'${flood_exp/1e9:.1f}B\n({flood_exp/total_exp*100:.0f}%)', '#1976D2'),
    ('AVG ANNUAL LOSS', f'${aal/1e6:.0f}M', '#E65100'),
    ('100-YEAR EVENT LOSS', f'${loss_100yr/1e9:.1f}B', '#BF360C'),
    ('500-YEAR EVENT LOSS', f'${loss_500yr/1e9:.1f}B', '#7B1FA2'),
]

y_pos = 0.95
ax_kpi.set_title('Panel B — Key Risk Metrics', fontsize=12)
for label, value, color in kpi_data:
    ax_kpi.text(0.05, y_pos, label, transform=ax_kpi.transAxes,
                fontsize=8, color='gray', fontweight='bold')
    ax_kpi.text(0.05, y_pos - 0.07, value, transform=ax_kpi.transAxes,
                fontsize=13, color=color, fontweight='bold')
    ax_kpi.axhline(y=y_pos - 0.12, xmin=0.05, xmax=0.95,
                   color='#EEEEEE', linewidth=1, transform=ax_kpi.transAxes)
    y_pos -= 0.19

# ── Panel C: EP Curve ────────────────────────────────────────────────────
ax_ep = fig.add_subplot(gs[1, 0])

rp_points = np.array([10, 25, 50, 100, 200, 500])
coeffs = np.polyfit(np.log([100, 500]), [loss_100yr, loss_500yr], 1)
losses_interp = np.clip(np.polyval(coeffs, np.log(rp_points)), 0, None)

ax_ep.fill_between(rp_points, 0, losses_interp/1e9, alpha=0.12, color='steelblue')
ax_ep.plot(rp_points, losses_interp/1e9, '-', color='steelblue', linewidth=2.5)
ax_ep.plot([100, 500], [loss_100yr/1e9, loss_500yr/1e9],
           'o', color='navy', markersize=9, zorder=6, label='Modeled scenarios')
ax_ep.axhline(y=1.0, color='#E53935', linestyle='--', linewidth=1.8,
              label='March 2026 event (~\$1B)')

for rp, loss in zip([100, 500], [loss_100yr, loss_500yr]):
    ax_ep.annotate(f'\${loss/1e9:.1f}B',
                   xy=(rp, loss/1e9), xytext=(rp * 0.55, loss/1e9 + 0.7),
                   fontsize=9, color='navy', fontweight='bold')

ax_ep.set_xlabel('Return Period (years)', fontsize=10)
ax_ep.set_ylabel('Loss (USD Billions)', fontsize=10)
ax_ep.set_title('Panel C — Exceedance\nProbability Curve', fontsize=12)
ax_ep.set_xscale('log')
ax_ep.set_xticks([10, 25, 50, 100, 200, 500])
ax_ep.set_xticklabels(['10', '25', '50', '100', '200', '500'])
ax_ep.legend(fontsize=8)
ax_ep.grid(True, alpha=0.3)

# ── Panel D: Flood depth by location ─────────────────────────────────────
ax_depth = fig.add_subplot(gs[1, 1])

df_sorted = df_haz[df_haz['return_period'] == 100].sort_values('depth_m', ascending=True)
colors_depth = ['#E3F2FD' if i == 'Oahu' else '#FFF3E0' for i in df_sorted['island']]
bars = ax_depth.barh(df_sorted['location'], df_sorted['depth_m'],
                     color=colors_depth, edgecolor='steelblue', linewidth=0.8)

for bar, val in zip(bars, df_sorted['depth_m']):
    ax_depth.text(val + 0.03, bar.get_y() + bar.get_height()/2,
                  f'{val:.1f}m', va='center', fontsize=8)

oahu_patch = mpatches.Patch(color='#E3F2FD', edgecolor='steelblue', label='Oahu')
maui_patch = mpatches.Patch(color='#FFF3E0', edgecolor='steelblue', label='Maui')
ax_depth.legend(handles=[oahu_patch, maui_patch], fontsize=8)
ax_depth.set_xlabel('Flood Depth — 100yr Event (m)', fontsize=10)
ax_depth.set_title('Panel D — Assigned Flood Depths\nby Location (100-Year)', fontsize=12)
ax_depth.grid(True, alpha=0.3, axis='x')
ax_depth.tick_params(axis='y', labelsize=8)

# ── Panel E: Protection gap ───────────────────────────────────────────────
ax_gap = fig.add_subplot(gs[1, 2])

NFIP_PENETRATION = 0.20
insured   = loss_100yr * NFIP_PENETRATION
uninsured = loss_100yr * (1 - NFIP_PENETRATION)

cats   = ['Total\nEconomic Loss', 'Insured Loss\n(NFIP ~20%)', 'Protection\nGap (80%)']
vals   = [loss_100yr/1e9, insured/1e9, uninsured/1e9]
colors_gap = ['#1565C0', '#2E7D32', '#C62828']

b = ax_gap.bar(cats, vals, color=colors_gap, width=0.55,
               edgecolor='white', linewidth=1.5)
for bar, val in zip(b, vals):
    ax_gap.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f'\${val:.1f}B', ha='center', fontsize=11, fontweight='bold')

ax_gap.set_ylabel('USD Billions', fontsize=10)
ax_gap.set_title('Panel E — Insurance Protection Gap\n100-Year Event', fontsize=12)
ax_gap.set_ylim(0, loss_100yr/1e9 * 1.25)
ax_gap.grid(True, alpha=0.3, axis='y')
ax_gap.text(0.5, -0.2,
            'NFIP penetration rate ~20% (Source: FEMA)',
            transform=ax_gap.transAxes, ha='center', fontsize=8, color='gray')

plt.savefig('fig2_hawaii_flood_risk_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: fig2_hawaii_flood_risk_dashboard.png')

---
## 8. Insurance Protection Gap

In [ ]:
# NFIP penetration rates for Hawaii (source: FEMA policy statistics)
# Hawaii has one of the lower NFIP take-up rates nationally
# Approximately 20% of properties in high-risk zones carry flood insurance

NFIP_PENETRATION = 0.20

scenarios = {
    '100-year event': loss_100yr,
    '500-year event': loss_500yr,
}

print('=' * 62)
print('  INSURANCE PROTECTION GAP ANALYSIS')
print('  NFIP Penetration Assumption: ~20% of at-risk properties')
print('=' * 62)

for scenario, gross_loss in scenarios.items():
    insured_loss   = gross_loss * NFIP_PENETRATION
    uninsured_loss = gross_loss * (1 - NFIP_PENETRATION)
    print(f'\n  {scenario}:')
    print(f'    Gross economic loss:    ${gross_loss/1e9:>8.2f}B')
    print(f'    Estimated insured:      ${insured_loss/1e9:>8.2f}B  ({NFIP_PENETRATION*100:.0f}%)')
    print(f'    Protection gap:         ${uninsured_loss/1e9:>8.2f}B  ({(1-NFIP_PENETRATION)*100:.0f}%)')

print('\n' + '=' * 62)
print('  KEY INSIGHT')
print('  The March 2026 Kona Low caused ~$1B in reported damage.')
print('  This falls well below the modeled 100-year threshold,')
print('  consistent with Governor Green\'s description of the event')
print('  as the worst flooding in 20 years (~1-in-20 year frequency).')
print('  The 100-year tail risk is estimated at ~20x the March event.')
print('=' * 62)

---
## 9. Limitations & Model Caveats

This analysis is an **ad-hoc, rapid-assessment model** intended for exploratory and portfolio purposes. The following limitations should be understood before drawing operational conclusions:

### 9.1 Hazard
- **22 manually placed flood zone points** represent the hazard footprint. A production model would use a continuous, physically simulated flood inundation surface.
- **Flood depths are assumed**, not modeled. Depths were assigned based on FEMA flood zone classification (AE, VE, X) rather than derived from hydraulic simulation.
- **Kona Low ≠ River Flood.** The March 2026 event was driven by extreme rainfall on steep terrain — flash flooding rather than river overflow. The JRC depth-damage functions were calibrated for slow-onset river flooding and may not accurately represent flash flood damage mechanics.
- **ISIMIP global flood data has no signal for Hawaii.** The global river routing model requires large basin drainage areas. Hawaii's watersheds are too small for this framework.
- **500-year depths are estimated** as 1.4× the 100-year depths — a simplification that should be replaced with a proper extreme value distribution in a production setting.

### 9.2 Exposure
- **LitPop is a proxy**, not a property database. It distributes GDP-calibrated asset values spatially using nighttime lights and population — it does not distinguish building types, occupancy, or construction quality.
- **No replacement cost data.** Insured values would require actual policy records or a replacement cost model.

### 9.3 Impact
- **The EP curve has only two anchor points** (100yr and 500yr). A robust EP curve requires a full stochastic event set with hundreds to thousands of simulated events.
- **NFIP penetration rate** of 20% is an estimate based on publicly available FEMA statistics. Actual penetration varies significantly by zip code and property type.

### 9.4 What This Model Is Good For
Despite these limitations, the analysis is useful for:
- Establishing **order-of-magnitude** loss estimates for Hawaii flood exposure
- Demonstrating the **insurance protection gap** concept with real geographic context
- Illustrating the **cat modeling workflow** (Hazard → Exposure → Vulnerability → Impact)
- Contextualizing the March 2026 Kona Low event against longer return-period benchmarks

---
## 10. Executive Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           HAWAII FLOOD RISK — EXECUTIVE SUMMARY                     ║
║           Oahu & Maui | Post Kona Low Event | April 2026            ║
╠══════════════════════════════════════════════════════════════════════╣
║  CONTEXT                                                             ║
║  The March 2026 Kona Low struck Hawaii with wind gusts of           ║
║  70-100 mph, causing the worst flooding in 20+ years. Damage        ║
║  estimates exceed $1 billion across public and private sectors.     ║
║  This analysis quantifies the broader tail-risk exposure.           ║
╠══════════════════════════════════════════════════════════════════════╣
║  METHODOLOGY                                                         ║
║  Platform:    CLIMADA open-source cat modeling (ETH Zurich)         ║
║  Hazard:      FEMA FIRM flood zone classifications (Oahu, Maui)     ║
║  Exposure:    LitPop produced capital, 2020 (CLIMADA Data API)      ║
║  Vuln. fn:    JRC North America depth-damage curve (RF4)            ║
║  Scenarios:   100-year (1% AEP) and 500-year (0.2% AEP)            ║
╠══════════════════════════════════════════════════════════════════════╣
║  EXPOSURE SUMMARY                                                    ║"""
f"""
║  Total Hawaii asset value:          ${total_exp/1e9:>8.1f} billion USD          ║
║  Assets in FEMA flood zones:        ${flood_exp/1e9:>8.1f} billion USD          ║
║  Flood zone share of total:         {flood_exp/total_exp*100:>8.1f}%                     ║
╠══════════════════════════════════════════════════════════════════════╣
║  MODELED LOSSES                                                      ║
║  Average Annual Loss (AAL):         ${aal/1e6:>8.0f} million USD         ║
║  100-year event gross loss:         ${loss_100yr/1e9:>8.2f} billion USD          ║
║  500-year event gross loss:         ${loss_500yr/1e9:>8.2f} billion USD          ║
╠══════════════════════════════════════════════════════════════════════╣
║  INSURANCE PROTECTION GAP (100-year)                                 ║
║  Estimated insured loss (~20% NFIP):{loss_100yr*0.2/1e9:>8.2f} billion USD          ║
║  Uninsured loss:                    ${loss_100yr*0.8/1e9:>8.2f} billion USD          ║
║  Uninsured share:                          80%                       ║
╠══════════════════════════════════════════════════════════════════════╣
║  KEY FINDINGS                                                        ║
║  1. The March 2026 Kona Low (~$1B) falls well below the modeled     ║
║     100-year loss (~$20B), suggesting a ~1-in-20 year frequency     ║
║     — consistent with the governor's 'worst in 20 years' framing.  ║
║  2. 80% of economic flood losses in Hawaii are uninsured, driven    ║
║     by low NFIP take-up rates and limited private flood market.     ║
║  3. The $16B protection gap on a 100-year event represents a        ║
║     significant opportunity for specialty flood insurers.           ║
╠══════════════════════════════════════════════════════════════════════╣
║  LIMITATIONS                                                         ║
║  Ad-hoc model. Hazard based on 22 FEMA zone points with assumed     ║
║  depths. LitPop is a GDP proxy, not a policy database. EP curve     ║
║  interpolated from two scenarios only. Results are order-of-        ║
║  magnitude estimates, not production model outputs.                 ║
╚══════════════════════════════════════════════════════════════════════╝
""")

---
## References

- Aznar-Siguan, G. and Bresch, D. N. (2019): CLIMADA v1: a global weather and climate risk assessment platform. *Geosci. Model Dev.*, 12, 3085–3097.
- Bresch, D. N. and Aznar-Siguan, G. (2021): CLIMADA v1.4.1: towards a globally consistent adaptation options appraisal tool. *Geosci. Model Dev.*, 14, 351–363.
- Huizinga, J., Moel, H. de and Szewczyk, W. (2017): Global flood depth-damage functions: Methodology and the Database with Guidelines. Joint Research Centre (JRC). doi: 10.2760/16510.
- FEMA (2021): Flood Hazard Areas (DFIRM) — State of Hawaii. National Flood Hazard Layer.
- Hawaii Emergency Management Agency (2026): March 2026 Kona Low Storm Recovery. https://dod.hawaii.gov/hiema/march-2026-kona-low/
- Governor Josh Green (2026): Hawaii flooding press conference. CNN, March 21, 2026.